# Introduction

This notebook presents a synthetic data generation for a program portfolio designed for a Doctor of Business Administration (DBA) research project. The objective of this simulation is to model a structured program environment composed of multiple projects and activities, each evaluated across standardized criteria on a 1–5 scale.

The generated dataset will serve as the foundation for subsequent analytical steps, including:

- The definition and implementation of aggregation rules;
- The application of machine learning models on the aggregated outputs.

It is designed to reflect realistic organizational decision-making conditions, where projects are assessed based on strategic alignment, complexity, risk, value, and other governance-related dimensions. To ensure full reproducibility, all stochastic processes will be controlled using a fixed random seed.

## 1-Data Loading and Overview

In [1]:
import pandas as pd
import numpy as np

# Do not display warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# Fixed random seed for full reproducibility
np.random.seed(42)

# Simulation Dataset import and structural overview of all sheets
file_path = "/kaggle/input/datasets/jfjutras07/data-generation-dataset/1-Data_Generation_Dataset.xlsx"

xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names

dataframes = {}

for sheet in sheet_names:
    dataframes[sheet] = pd.read_excel(xls, sheet_name=sheet)

overview = pd.DataFrame({
    "table_name": sheet_names,
    "n_rows": [dataframes[s].shape[0] for s in sheet_names],
    "n_cols": [dataframes[s].shape[1] for s in sheet_names]
})

overview

,table_name,n_rows,n_cols
0,Activity,196,7
1,Activity_Certainty_Score,196,7
2,Partner,5,2
3,Project_Certainty_Score,50,14
4,Project_Partner,390,14
5,Project_Volunteer,78,7


## 2-Data Model Understanding

The dataset used in this notebook represents a subset of the complete simulation environment developed for the validation of the proposed decision-support framework. It does not include all theoretical, structural, or analytical components of the complete simulation model. Instead, it contains only the data structures required at this stage to generate and organize simulated evaluation scores and associated credibility information.

The dataset was constructed from the data requirements identified in the evaluation framework and represents the information that would result from the collection of activity-level and project-level assessments. All records, identifiers, and entities included in this dataset are artificially generated and anonymized. The purpose is not to reproduce an existing organizational database, but to provide a controlled and coherent environment for testing downstream aggregation rules and analytical approaches.

Additional simulation components related to program structure, contextual information, and analytical processes will be progressively introduced in subsequent notebooks when required by the validation objectives.

### Key methodological characteristics

The system is based on anchored ordinal scoring (1–5) applied across two distinct constructs:

- **Ordinal project and activity scores**: represent evaluated characteristics of activities and projects.
- **Credibility scores (CS)**: represent the confidence in the evidence supporting each measurement.

Both score types are derived from structured evaluation processes and must be treated as distinct dimensions in the simulation.

### Dataset Scope

The current dataset includes only the tables required to:

- Generate simulated activity-level performance and credibility scores;
- Generate project-level credibility structures;
- Represent project–partner relationships;
- Support multi-rater evaluation simulations.

Other elements of the complete simulation model are intentionally excluded at this stage to maintain clarity and focus on the validation of the scoring, aggregation, and analytical mechanisms.

## 3-Research Methodology

The following table presents the evaluation structure used in the simulation dataset, including the assessed components, evaluation sources, measurement dimensions, and scoring approach.

| Level | Component | Description | Variables / Dimensions | Measurement Method | Scale | Notes |
|------|----------|-------------|------------------------|--------------------|-------|------|
| Activity | Activity | Core unit of analysis within projects | Cost, Time, Novelty, Effort | Individual interviews (volunteers) | Anchored ordinal (1–5) | Activity scores |
| Activity | Activity CS | Credibility Score for activity-level evaluation | Evidence quality, documentation strength, source availability (cost, time, novelty, effort) | Analyst assessment | Anchored ordinal (1–5) | Independent from activity scores |
| Project | Project | Project-Level Evaluation | Use, Quality, Reach, Results, Satisfaction, Alignment, Durability, Resilience | Group interviews (partners)| Anchored ordinal (1–5) | Project Scores |
| Project | Project | Project-Level Evaluation | Collaboration, Stability | Individual interviews (volunteers) | Anchored ordinal (1–5) | Project Scores |
| Project | Project CS | Credibility Score for project-level evaluation | Evidence quality, documentation strength, source availability (use, quality, reach, results, satisfaction, alignment, durability, resilience, collaboration, stability) | Analyst assessment | Anchored ordinal (1–5) | Independent from project scores |

## 4-Structural Integrity Check

This section ensures that all relational structures between tables are valid before generating simulated values.

The focus is strictly on structural consistency:
- existence of referenced keys across tables
- integrity of project–partner relationships
- alignment of activity and project identifiers

No statistical validation or missing data analysis is performed at this stage, as data completeness will be enforced during the simulation phase.

In [2]:
#Dataframe Naming
df_activity = dataframes["Activity"]
df_activity_cs = dataframes["Activity_Certainty_Score"]
df_partner = dataframes["Partner"]
df_project_cs = dataframes["Project_Certainty_Score"]
df_project_partner = dataframes["Project_Partner"]
df_project_volunteer = dataframes["Project_Volunteer"]

#Structural Integrity Checks
project_ids = set(df_project_cs["id_project"])
activity_ids = set(df_activity["id_project"])

missing_activity = project_ids - activity_ids

pp_projects = set(df_project_partner["id_project"])
missing_pp = project_ids - pp_projects

partner_ids = set(df_partner["id_partner"])
pp_partners = set(df_project_partner["id_partner"])
invalid_partners = pp_partners - partner_ids

pv_projects = set(df_project_volunteer["id_project"])
missing_pv = project_ids - pv_projects

#Output
{
    "missing_activity": list(missing_activity),
    "missing_project_partner": list(missing_pp),
    "invalid_partners": list(invalid_partners),
    "missing_project_volunteer": list(missing_pv)
}

{'missing_activity': [],
 'missing_project_partner': [],
 'invalid_partners': [8, 6, 7],
 'missing_project_volunteer': []}

In [3]:
# Force correction on Excel rows 305–308
df_project_partner.loc[305:309, "id_partner"] = 4

# Verification
df_project_partner.loc[305:309]

,id_project,Projects,id_partner,id_rater,project_type,project_class,use,quality,reach,results,satisfaction,alignment,durability,resilience
305,43,Daily Operations Management,4,16,Capacity Building,Internal - Behavioral,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
306,44,Daily Operations Management,4,17,Capacity Building,Internal - Behavioral,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
307,45,Daily Operations Management,4,18,Capacity Building,Internal - Behavioral,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
308,46,Daily Operations Management,4,19,Capacity Building,Internal - Behavioral,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
309,47,Daily Operations Management,4,20,Capacity Building,Internal - Behavioral,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**A minor data entry inconsistency was identified in the initial simulation file and corrected before data generation to ensure referential integrity across all tables.**

In [4]:
# Quick validation
project_ids = set(df_project_cs["id_project"])

activity_ids = set(df_activity["id_project"])
missing_activity = project_ids - activity_ids

pp_projects = set(df_project_partner["id_project"])
missing_pp = project_ids - pp_projects

partner_ids = set(df_partner["id_partner"])
pp_partners = set(df_project_partner["id_partner"])
invalid_partners = pp_partners - partner_ids

pv_projects = set(df_project_volunteer["id_project"])
missing_pv = project_ids - pv_projects

{
    "missing_activity": list(missing_activity),
    "missing_project_partner": list(missing_pp),
    "invalid_partners": list(invalid_partners),
    "missing_project_volunteer": list(missing_pv)
}

{'missing_activity': [],
 'missing_project_partner': [],
 'invalid_partners': [],
 'missing_project_volunteer': []}

## 5-Activity Data Generation

The simulation environment was constructed from a reconstructed program portfolio inspired by international development program structures observed in post-closure contexts (in this case VCP Guatemala). The simulated portfolio includes multiple projects implemented by individual partners and through multi-partner collaborations, covering a variety of organizational development, capacity-building, assessment, and operational activities.

The purpose of this simulation is not to reproduce a specific program or estimate actual project performance. Instead, the portfolio structure provides a controlled environment for testing the proposed aggregation mechanisms and analytical approaches. Project types, activity profiles, partnership structures, and evaluation dimensions were defined to generate heterogeneous conditions representative of complex program portfolios.

The generation process follows predefined probabilistic business rules derived from the evaluation framework. These rules define expected tendencies across activity characteristics, project dimensions, and credibility scores while preserving random variation. Therefore, simulated values are not deterministic outputs but probabilistic realizations designed to maintain variability and uncertainty similar to real-world evaluation settings.

In [5]:
# Generate ordinal scores from predefined probability distributions.
# The same function is used for both performance scores and credibility scores,
# as both follow the same anchored ordinal scale (1–5) with probabilistic sampling.
def sample_ordinal_score(dist):
    values = [1, 2, 3, 4, 5]
    dist = np.array(dist)
    return np.random.choice(values, p=dist / dist.sum())

# Define activity simulation profiles
activity_profiles = pd.DataFrame({
    "activity_type": [
        "Document Analysis",
        "Document Writing",
        "Interview",
        "Focus Group",
        "Forum",
        "Mapping",
        "Observation",
        "Presentation",
        "Survey",
        "Technical Assistance",
        "Technical Development",
        "Training",
        "Workshop",
        "Coaching"
    ],
    "cost": [
        "low","low","low","moderate","high","low","low","moderate",
        "low","moderate","high","moderate","high","moderate"
    ],
    "time": [
        "moderate","moderate","moderate","moderate","moderate","moderate","short","short",
        "moderate","short","moderate","moderate","moderate","moderate"
    ],
    "novelty": [
        "very_low","moderate","moderate","moderate","moderate","moderate","moderate","moderate",
        "moderate","moderate","moderate","moderate","moderate","moderate"
    ],
    "effort": [
        "moderate","moderate","moderate","moderate","moderate","moderate","low","low",
        "moderate","moderate","moderate","moderate","moderate","moderate"
    ]
})

In [6]:
# Define ordinal probability distributions for simulation profiles
ordinal_distributions = {
    "very_low": [0.60, 0.30, 0.10, 0.00, 0.00],
    "low":      [0.40, 0.40, 0.15, 0.05, 0.00],
    "moderate": [0.15, 0.30, 0.30, 0.20, 0.05],
    "high":     [0.00, 0.10, 0.35, 0.40, 0.15],
    "short":    [0.50, 0.30, 0.15, 0.05, 0.00]
}

In [7]:
# Utility functions for probability normalization and ordinal score sampling.
def normalize(dist):
    dist = np.array(dist, dtype=float)
    return dist / dist.sum()

# Generate activity ordinal scores from simulation profiles.
def sample_activity_score(profile, dimension, complexity_signal=0):

    # Base distribution from predefined activity profiles
    base_dist = np.array(ordinal_distributions[profile])

    # Define ordinal scale depending on dimension
    if dimension in ["cost", "time", "effort"]:
        values = [1, 2, 3, 4]
        base_dist = base_dist[:4]  # Truncate novelty dimension
    else:
        values = [1, 2, 3, 4, 5]

    # Weak complexity adjustment: high complexity activities slightly shift probability mass upward
    # without creating deterministic relationships
    adjustment = complexity_signal * 0.03
    base_dist = base_dist + adjustment

    # Ensure valid probability distribution
    base_dist = np.clip(base_dist, 0.001, None)
    dist = normalize(base_dist)

    return np.random.choice(values, p=dist)

In [8]:
# Precompute lookup dictionary for activity profiles
profiles = activity_profiles.set_index("activity_type").to_dict("index")

# Generate scores
for i, row in df_activity.iterrows():
    profile = profiles[row["activity_type"]]

    # Simple complexity proxy based on activity characteristics
    complexity_signal = (
        (profile["cost"] == "high") +
        (profile["effort"] == "high")
    )

    df_activity.loc[i, "cost"] = sample_activity_score(profile["cost"], "cost", complexity_signal)
    df_activity.loc[i, "time"] = sample_activity_score(profile["time"], "time", complexity_signal)
    df_activity.loc[i, "novelty"] = sample_activity_score(profile["novelty"], "novelty", complexity_signal)
    df_activity.loc[i, "effort"] = sample_activity_score(profile["effort"], "effort", complexity_signal)

# Force correct dtype after simulation
df_activity[["cost", "time", "novelty", "effort"]] = df_activity[
    ["cost", "time", "novelty", "effort"]
].astype(int)

In [9]:
# Validate that all generated scores are within the valid ordinal range (1–5).
assert df_activity[["cost", "time", "novelty", "effort"]].apply(
    lambda col: col.between(1, 5)
).all().all()

df_activity[["cost", "time", "novelty", "effort"]].describe()

,cost,time,novelty,effort
count,196.000000,196.000000,196.000000,196.000000
mean,2.193878,2.433673,2.352041,2.525510
std,1.054010,1.002913,1.169504,1.049719
min,1.000000,1.000000,1.000000,1.000000
25%,1.000000,2.000000,1.000000,2.000000
50%,2.000000,2.000000,2.000000,3.000000
75%,3.000000,3.000000,3.000000,3.000000
max,4.000000,4.000000,5.000000,4.000000


## 6-CS Activity-Level Data Generation

Activity credibility scores are generated using predefined probabilistic rules based on the type of activity and the expected availability of supporting evidence. These rules define different credibility across cost, time, novelty, and effort dimensions while preserving random variation between activities.

In [10]:
# Generate activity-level credibility scores based on predefined probability distributions.
# Define credibility score distributions by activity type and dimension.
for i, row in df_activity.iterrows():
    activity_type = row["activity_type"]

    # CS_cost: generally higher credibility due to documentation availability
    if activity_type in ["Forum", "Presentation", "Workshop"]:
        dist_cost = [0.05, 0.10, 0.25, 0.35, 0.25]
    else:
        dist_cost = [0.10, 0.25, 0.35, 0.20, 0.10]

    # CS_time: generally low credibility, except for structured projects
    dist_time = [0.45, 0.35, 0.15, 0.05, 0.00]

    # CS_novelty: mixed evidence availability depending on outputs
    if activity_type == "Document Analysis":
        dist_novelty = [0.30, 0.35, 0.25, 0.10, 0.00]
    else:
        dist_novelty = [0.10, 0.20, 0.30, 0.25, 0.15]

    # CS_effort: generally low credibility due to interview-based evidence
    dist_effort = [0.50, 0.30, 0.15, 0.05, 0.00]

    df_activity.loc[i, "CS_cost"] = sample_ordinal_score(dist_cost)
    df_activity.loc[i, "CS_time"] = sample_ordinal_score(dist_time)
    df_activity.loc[i, "CS_novelty"] = sample_ordinal_score(dist_novelty)
    df_activity.loc[i, "CS_effort"] = sample_ordinal_score(dist_effort)

# Force correct dtype after simulation
df_activity[["CS_cost", "CS_time", "CS_novelty", "CS_effort"]] = df_activity[
    ["CS_cost", "CS_time", "CS_novelty", "CS_effort"]
].astype(int)

# Validate credibility scores are within valid ordinal range (1–5)
assert df_activity[["CS_cost", "CS_time", "CS_novelty", "CS_effort"]].apply(
    lambda col: col.between(1, 5)
).all().all()

df_activity[["CS_cost", "CS_time", "CS_novelty", "CS_effort"]].describe()

,CS_cost,CS_time,CS_novelty,CS_effort
count,196.000000,196.000000,196.000000,196.000000
mean,3.224490,1.806122,2.892857,1.765306
std,1.198551,0.830917,1.237761,0.856960
min,1.000000,1.000000,1.000000,1.000000
25%,2.000000,1.000000,2.000000,1.000000
50%,3.000000,2.000000,3.000000,2.000000
75%,4.000000,2.000000,4.000000,2.000000
max,5.000000,4.000000,5.000000,4.000000


## 7-Project-Level Data Generation (Partners)

Project-level performance scores are generated from partner assessments using predefined probability distributions. These distributions vary according to project characteristics and are applied independently across evaluation dimensions while preserving variability between projects and raters.

In [11]:
# Generate project-level performance scores using predefined probability distributions.
# Loop through each evaluation row
for i, row in df_project_partner.iterrows():
    project_class = row["project_class"]
    project_type = row["project_type"]

    # Use (strongly dependent on project class)
    if "Structural" in project_class:
        dist_use = [0.10, 0.35, 0.30, 0.20, 0.05]
    else:  # Behavioral
        dist_use = [0.15, 0.40, 0.30, 0.12, 0.03]

    # Quality (internal vs external evidence)
    if "External" in project_class:
        dist_quality = [0.10, 0.40, 0.30, 0.15, 0.05]
    else:
        dist_quality = [0.05, 0.30, 0.35, 0.25, 0.05]

    # Reach (broader variability)
    if project_type in ["Capacity Building", "Training", "Workshop"]:
        dist_reach = [0.10, 0.20, 0.30, 0.25, 0.15]
    else:
        dist_reach = [0.15, 0.25, 0.35, 0.20, 0.05]

    # Results (dependent on structural vs behavioral)
    if "Structural" in project_class:
        dist_results = [0.10, 0.30, 0.30, 0.20, 0.10]
    else:
        dist_results = [0.20, 0.35, 0.30, 0.12, 0.03]

    # Satisfaction (mostly moderate, interview-based)
    dist_satisfaction = [0.10, 0.35, 0.35, 0.15, 0.05]

    # Alignment (moderate-high tendency)
    dist_alignment = [0.05, 0.20, 0.35, 0.30, 0.10]

    # Durability (higher for structural projects)
    if "Structural" in project_class:
        dist_durability = [0.05, 0.20, 0.30, 0.30, 0.15]
    else:
        dist_durability = [0.10, 0.30, 0.35, 0.20, 0.05]

    # Resilience (generally high, rare low values)
    dist_resilience = [0.05, 0.10, 0.25, 0.35, 0.25]

    df_project_partner.loc[i, "use"] = sample_ordinal_score(dist_use)
    df_project_partner.loc[i, "quality"] = sample_ordinal_score(dist_quality)
    df_project_partner.loc[i, "reach"] = sample_ordinal_score(dist_reach)
    df_project_partner.loc[i, "results"] = sample_ordinal_score(dist_results)
    df_project_partner.loc[i, "satisfaction"] = sample_ordinal_score(dist_satisfaction)
    df_project_partner.loc[i, "alignment"] = sample_ordinal_score(dist_alignment)
    df_project_partner.loc[i, "durability"] = sample_ordinal_score(dist_durability)
    df_project_partner.loc[i, "resilience"] = sample_ordinal_score(dist_resilience)

# Validate score ranges
assert df_project_partner[
    ["use","quality","reach","results","satisfaction","alignment","durability","resilience"]
].apply(lambda col: col.between(1,5)).all().all()

df_project_partner[[
    "use","quality","reach","results","satisfaction","alignment","durability","resilience"
]].describe()

,use,quality,reach,results,satisfaction,alignment,durability,resilience
count,390.000000,390.000000,390.000000,390.000000,390.000000,390.000000,390.000000,390.000000
mean,2.743590,2.853846,2.815385,2.700000,2.653846,3.284615,3.102564,3.564103
std,1.009415,0.989234,1.187670,1.106071,0.962363,1.013289,1.101410,1.160509
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,2.000000,2.000000,2.000000,2.000000,2.000000,3.000000,2.000000,3.000000
50%,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,4.000000
75%,3.000000,4.000000,4.000000,3.000000,3.000000,4.000000,4.000000,4.000000
max,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


## 8-Project-Level Data Generation (Volunteers)

Project-level collaboration and stability scores are generated from volunteer assessments using predefined probability distributions. These distributions consider project structure and complexity while preserving variability between projects and respondents.

In [12]:
# Generate project-level collaboration and stability scores using predefined probability distributions.
# Compute activity count per project (complexity proxy)
activity_count = df_project_volunteer.groupby("id_project").size().to_dict()

# Loop through each evaluation row
for i, row in df_project_volunteer.iterrows():
    project_class = row["project_class"]
    project_type = row["project_type"]
    n_activities = activity_count[row["id_project"]]

    # Collaboration (interview-based, moderate variability)
    # Mostly 2–4, with rare extremes; slightly more difficulty as complexity increases
    if n_activities <= 5:
        dist_collaboration = [0.05, 0.25, 0.40, 0.25, 0.05]
    elif n_activities <= 10:
        dist_collaboration = [0.05, 0.30, 0.40, 0.20, 0.05]
    else:
        dist_collaboration = [0.08, 0.35, 0.35, 0.18, 0.04]

    # Stability (internal risk impact: higher score = more stable)
    # Strong skew toward 4–5, 3 uncommon, 1–2 exceptional only
    # Structural projects slightly more stable due to documentation and systems

    if "Structural" in project_class:
        dist_stability = [0.005, 0.02, 0.08, 0.35, 0.535]
    else:
        dist_stability = [0.01, 0.04, 0.12, 0.38, 0.45]

    # Ensure behavioral projects remain slightly more volatile
    if "Behavioral" in project_class:
        dist_collaboration = np.array(dist_collaboration)
        dist_collaboration = dist_collaboration * np.array([1.1, 1.05, 1.0, 0.95, 0.9])
        dist_collaboration = dist_collaboration / dist_collaboration.sum()

    df_project_volunteer.loc[i, "collaboration"] = sample_ordinal_score(dist_collaboration)
    df_project_volunteer.loc[i, "stability"] = sample_ordinal_score(dist_stability)

# Validate score ranges
assert df_project_volunteer[
    ["collaboration", "stability"]
].apply(lambda col: col.between(1, 5)).all().all()

df_project_volunteer[
    ["collaboration", "stability"]
].describe()

,collaboration,stability
count,78.000000,78.000000
mean,2.807692,4.230769
std,0.912506,0.804582
min,1.000000,1.000000
25%,2.000000,4.000000
50%,3.000000,4.000000
75%,3.000000,5.000000
max,5.000000,5.000000


## 9-CS Project-Level Data Generation

Project-level credibility scores are generated using predefined probability distributions based on project characteristics and expected evidence availability. These scores represent confidence in the supporting evidence rather than project performance. Variability is preserved across projects to maintain differences in evidence quality and assessment uncertainty.

In [13]:
# Generate project-level credibility scores using predefined probability distributions.
for i, row in df_project_cs.iterrows():
    project_class = row["project_class"]
    project_type = row["project_type"]

    # CS_collaboration (documented coordination quality across partners)
    # Slightly lower when external/structural due to weaker shared documentation alignment
    if "External" in project_class:
        dist_collaboration = [0.05, 0.20, 0.35, 0.30, 0.10]
    else:
        dist_collaboration = [0.03, 0.15, 0.35, 0.35, 0.12]

    # CS_stability (internal risk evidence; high values dominate, low values rare)
    # Strong bias toward high stability when documentation is complete and structured
    if "Structural" in project_class:
        dist_stability = [0.00, 0.03, 0.12, 0.45, 0.40]
    else:
        dist_stability = [0.00, 0.05, 0.15, 0.45, 0.35]

    # CS_use (clarity and usability of documented outputs)
    dist_use = [0.05, 0.20, 0.35, 0.30, 0.10] if "Structural" in project_class else [0.08, 0.25, 0.35, 0.25, 0.07]

    # CS_quality (rigor and completeness of documentary evidence)
    dist_quality = [0.03, 0.18, 0.35, 0.32, 0.12] if "Structural" in project_class else [0.05, 0.22, 0.35, 0.28, 0.10]

    # CS_reach (breadth of documentation coverage across stakeholders)
    dist_reach = [0.05, 0.20, 0.35, 0.30, 0.10]

    # CS_results (documented outcomes strength, moderately conservative distribution)
    dist_results = [0.05, 0.20, 0.35, 0.30, 0.10]

    # CS_satisfaction (interview-based perception of evidence adequacy)
    dist_satisfaction = [0.05, 0.25, 0.35, 0.25, 0.10]

    # CS_alignment (coherence between documentary evidence and project objectives)
    dist_alignment = [0.03, 0.15, 0.35, 0.35, 0.12]

    # CS_durability (sustainability of documented outputs over time)
    dist_durability = [0.02, 0.10, 0.28, 0.40, 0.20]

    # CS_resilience (external risk exposure on documented outcomes; high scores dominate)
    dist_resilience = [0.00, 0.05, 0.15, 0.40, 0.40]

    df_project_cs.loc[i, "CS_collaboration"] = sample_ordinal_score(dist_collaboration)
    df_project_cs.loc[i, "CS_stability"] = sample_ordinal_score(dist_stability)
    df_project_cs.loc[i, "CS_use"] = sample_ordinal_score(dist_use)
    df_project_cs.loc[i, "CS_quality"] = sample_ordinal_score(dist_quality)
    df_project_cs.loc[i, "CS_reach"] = sample_ordinal_score(dist_reach)
    df_project_cs.loc[i, "CS_results"] = sample_ordinal_score(dist_results)
    df_project_cs.loc[i, "CS_satisfaction"] = sample_ordinal_score(dist_satisfaction)
    df_project_cs.loc[i, "CS_alignment"] = sample_ordinal_score(dist_alignment)
    df_project_cs.loc[i, "CS_durability"] = sample_ordinal_score(dist_durability)
    df_project_cs.loc[i, "CS_resilience"] = sample_ordinal_score(dist_resilience)

# Validate score ranges (CS must stay within ordinal bounds 1–5)
assert df_project_cs[
    ["CS_collaboration","CS_stability","CS_use","CS_quality","CS_reach",
     "CS_results","CS_satisfaction","CS_alignment","CS_durability","CS_resilience"]
].apply(lambda col: col.between(1,5)).all().all()

df_project_cs[
    ["CS_collaboration","CS_stability","CS_use","CS_quality","CS_reach",
     "CS_results","CS_satisfaction","CS_alignment","CS_durability","CS_resilience"]
].describe()

,CS_collaboration,CS_stability,CS_use,CS_quality,CS_reach,CS_results,CS_satisfaction,CS_alignment,CS_durability,CS_resilience
count,50.000000,50.000000,50.000000,50.00000,50.000000,50.000000,50.000000,50.000000,50.00000,50.000000
mean,3.360000,4.180000,3.240000,3.20000,3.300000,3.180000,3.200000,3.360000,3.66000,4.180000
std,0.963836,0.719694,0.980629,0.92582,1.073807,1.137308,1.087968,0.963836,0.93917,0.918961
min,2.000000,2.000000,1.000000,2.00000,1.000000,1.000000,1.000000,1.000000,2.00000,2.000000
25%,3.000000,4.000000,3.000000,2.25000,3.000000,3.000000,3.000000,3.000000,3.00000,4.000000
50%,3.000000,4.000000,3.000000,3.00000,3.000000,3.000000,3.000000,3.000000,4.00000,4.000000
75%,4.000000,5.000000,4.000000,4.00000,4.000000,4.000000,4.000000,4.000000,4.00000,5.000000
max,5.000000,5.000000,5.000000,5.00000,5.000000,5.000000,5.000000,5.000000,5.00000,5.000000


## 10-Conclusion

This simulation provides a structured representation of a multi-level program evaluation system, including activities, projects, partners, and credibility scores. All variables are generated using predefined business rules and probability distributions while preserving controlled variability. The resulting dataset provides the foundation for subsequent analyses, including data modeling, aggregation strategies, and machine learning applications.

## 11-Data Export

In [14]:
import os

# Define export path
export_path = "/kaggle/working/"
excel_file = "simulation_dataset.xlsx"
excel_path = os.path.join(export_path, excel_file)

# Ensure directory exists
os.makedirs(export_path, exist_ok=True)

# Dictionary of all tables to export
tables_to_export = {
    "Activity": df_activity,
    "Activity_Certainty_Score": df_activity_cs,
    "Partner": df_partner,
    "Project_Certainty_Score": df_project_cs,
    "Project_Partner": df_project_partner,
    "Project_Volunteer": df_project_volunteer
}

# Write all tables into one Excel file (multi-sheet)
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, df in tables_to_export.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Excel file successfully created: {excel_path}")

Excel file successfully created: /kaggle/working/simulation_dataset.xlsx
